# FS-UNeXt: Huấn luyện có giám sát UNeXt cho phân đoạn tế bào máu

**UNeXt**: CNN siêu nhẹ (~1.5M tham số) – mô hình được triển khai trong DACL.

Notebook này **độc lập** (không cần clone code), gồm:
1. Cấu hình thí nghiệm
2. Chuẩn bị & chia dữ liệu
3. **Tăng cường dữ liệu (Data Augmentation)** – cấu hình + xem trực quan
4. Dataset / DataLoader
5. Mô hình UNeXt
6. Hàm mất mát & chỉ số đánh giá
7. Vòng huấn luyện
8. Chạy thí nghiệm ở các mức nhãn
9. Kết quả: bảng, đường cong huấn luyện, ảnh dự đoán

**Trước khi chạy trên Kaggle**
- *Settings → Accelerator*: **GPU T4 x2** hoặc **GPU P100**
- *Settings → Internet*: **On** (không bắt buộc)
- *Add Input*: thêm Kaggle Dataset chứa bộ dữ liệu Blood Cell Segmentation (1328 ảnh + 1328 mask)
- Chạy toàn bộ: *Run All*, hoặc *Save Version → Save & Run All (Commit)* để chạy nền tới 12 giờ

## 1. Cấu hình thí nghiệm

In [ ]:
# ================================================================
#                      CẤU HÌNH THÍ NGHIỆM
#  Chỉ cần sửa trong ô này. Mọi ô phía dưới đọc cấu hình từ CFG.
# ================================================================
class CFG:
    # ---------------- ĐƯỜNG DẪN ----------------
    # Thư mục dữ liệu GỐC trên Kaggle (panel "Input" bên phải -> nút copy đường dẫn).
    # Script sẽ tự quét đệ quy và ghép cặp *_image / *_mask nên không cần đúng cấu trúc con.
    RAW_DIR = "/kaggle/input/blood-cell-segmentation"
    # Nơi ghi dữ liệu đã chuẩn hóa và kết quả. Trên Kaggle chỉ /kaggle/working được lưu lại.
    WORK_DIR = "/kaggle/working"

    # ---------------- MÔ HÌNH ----------------
    MODEL = "unext"
    # Cấu hình số kênh UNeXt:
    #   "dacl"     : (32, 64, 128, 160, 256) - như bài báo DACL khai báo   -> 1.58M tham số
    #   "original" : (16, 32, 128, 160, 256) - như bài báo UNeXt gốc        -> 1.47M tham số
    # (bài báo DACL ghi 1.47M nhưng khai báo số kênh của "dacl" -> mâu thuẫn).
    # Dùng CÙNG cấu hình cho FS-UNeXt và UNeXt trong DACL để so sánh công bằng.
    UNEXT_VARIANT = "dacl"
    NUM_CLASSES = 2          # 0 = nền, 1 = tế bào (mask nhị phân)

    # ---------------- DỮ LIỆU ----------------
    # Ảnh gốc 1600x1200 được thu nhỏ một lần về kích thước này (giữ tỉ lệ 4:3) để
    # DataLoader không phải đọc ảnh lớn ở mỗi vòng lặp -> nhanh hơn nhiều.
    PREP_SIZE = (640, 480)   # (rộng, cao)
    # Mask lưu dạng JPG (có nhiễu nén) -> pixel > ngưỡng là tế bào.
    MASK_THRESHOLD = 127
    # Kích thước ảnh đưa vào mạng. 224 tương thích Swin-Unet (window 7, trọng số ImageNet 224).
    # GIỮ NGUYÊN 224 cho cả UNeXt và Swin-Unet để so sánh công bằng.
    IMG_SIZE = 224
    # Tỉ lệ chia train/val/test và seed chia. GIỮ NGUYÊN seed giữa các notebook để mọi
    # mô hình (FS, DACL, baseline) dùng CHUNG một cách chia dữ liệu.
    SPLIT_RATIOS = (0.7, 0.1, 0.2)
    SPLIT_SEED = 2026
    # Các mức % ảnh có nhãn cần chạy. 100 = toàn bộ tập train (giới hạn trên).
    # 5/10/20 = chỉ dùng phần có nhãn tương ứng trong thí nghiệm SSL (giới hạn dưới của SSL).
    LABELED_RATIOS = [5, 10, 20, 100]

    # ---------------- HUẤN LUYỆN ----------------
    # Huấn luyện theo SỐ VÒNG LẶP (không theo epoch): số bước cập nhật như nhau ở mọi mức nhãn.
    MAX_ITERS = 30000
    # Kích thước batch. UNeXt rất nhẹ: 8 vừa thoải mái GPU 16GB (có thể tăng lên 16).
    BATCH_SIZE = 8
    # SGD theo bài báo DACL; learning rate giảm theo lịch poly: lr*(1 - it/max_it)^0.9.
    BASE_LR = 0.01
    MOMENTUM = 0.9
    WEIGHT_DECAY = 1e-4
    # Mixed precision FP16: nhanh hơn ~1.5-2 lần, tốn ít bộ nhớ GPU hơn.
    USE_AMP = True
    # Số tiến trình đọc dữ liệu (Kaggle có 4 CPU).
    NUM_WORKERS = 4
    # Danh sách seed. Thêm 2027, 2028 để chạy lặp và báo cáo mean ± std.
    SEEDS = [2026]

    # ---------------- ĐÁNH GIÁ & LOG ----------------
    # Đánh giá trên validation mỗi VAL_INTERVAL vòng; lưu best.pth khi Dice tốt hơn.
    VAL_INTERVAL = 500
    LOG_INTERVAL = 100

    # ---------------- CHẠY THỬ ----------------
    # True -> mỗi thí nghiệm chỉ chạy 300 vòng, để kiểm tra toàn bộ notebook trong vài phút.
    # NHỚ đặt lại False trước khi chạy thật.
    DEBUG = False


if CFG.DEBUG:
    CFG.MAX_ITERS, CFG.VAL_INTERVAL, CFG.LOG_INTERVAL = 300, 100, 50

In [ ]:
import csv, json, os, random, re, shutil, time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2
from scipy import ndimage
from skimage.color import hed2rgb, rgb2hed
from timm.layers import DropPath, to_2tuple, trunc_normal_
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = Path(CFG.WORK_DIR) / "data" / "processed"
SPLIT_DIR = Path(CFG.WORK_DIR) / "data" / "splits"
RUN_ROOT = Path(CFG.WORK_DIR) / "runs"
print("torch", torch.__version__, "| albumentations", A.__version__, "| device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Chuẩn bị & chia dữ liệu

- Ghép cặp ảnh/mask theo tên (`123_image.png` ↔ `123_mask.jpg`), thu nhỏ, nhị phân hóa mask.
- Chia **70/10/20** (train/val/test) với seed cố định; từ tập train lấy **5/10/20%** làm ảnh có nhãn.
- Ô này **tự bỏ qua** nếu dữ liệu đã được chuẩn bị trong phiên hiện tại.

> Kiểm tra log: phải ghép được **1328 cặp**, tỉ lệ pixel tế bào không quá gần 0 hoặc 1.

In [ ]:
EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
MASK_WORDS = ("mask", "gt", "label", "ground")
SUFFIX_RE = re.compile(r"[ _\-]?(mask|masks|gt|label|labels|image|images|img|original)$", re.I)


def _is_mask(p):
    return any(w in p.stem.lower() for w in MASK_WORDS) or any(w in p.parent.name.lower() for w in MASK_WORDS)


def prepare_dataset():
    if (DATA_DIR / "DONE").exists():
        print("Dữ liệu đã chuẩn hóa - bỏ qua.")
        return
    files = [p for p in Path(CFG.RAW_DIR).rglob("*") if p.suffix.lower() in EXTS]
    images, masks = {}, {}
    for p in files:
        (masks if _is_mask(p) else images)[SUFFIX_RE.sub("", p.stem).strip().lower()] = p
    keys = sorted(set(images) & set(masks))
    print(f"Tìm thấy {len(images)} ảnh, {len(masks)} mask -> ghép được {len(keys)} cặp")
    if not keys:
        raise RuntimeError("Không ghép được cặp nào - kiểm tra CFG.RAW_DIR và cách đặt tên file.")
    (DATA_DIR / "images").mkdir(parents=True, exist_ok=True)
    (DATA_DIR / "masks").mkdir(parents=True, exist_ok=True)
    fg = []
    for i, k in enumerate(tqdm(keys, desc="Chuẩn hóa")):
        img = cv2.imread(str(images[k]), cv2.IMREAD_COLOR)
        msk = cv2.imread(str(masks[k]), cv2.IMREAD_GRAYSCALE)
        if img is None or msk is None:
            print("  bỏ qua file lỗi:", k)
            continue
        img = cv2.resize(img, CFG.PREP_SIZE, interpolation=cv2.INTER_AREA)
        msk = cv2.resize(msk, CFG.PREP_SIZE, interpolation=cv2.INTER_NEAREST)
        msk = np.where(msk > CFG.MASK_THRESHOLD, 255, 0).astype(np.uint8)
        cv2.imwrite(str(DATA_DIR / "images" / f"{i:04d}.png"), img)
        cv2.imwrite(str(DATA_DIR / "masks" / f"{i:04d}.png"), msk)
        fg.append((msk > 0).mean())
    (DATA_DIR / "DONE").write_text("ok")
    print(f"Đã lưu {len(fg)} cặp | tỉ lệ pixel tế bào trung bình: {np.mean(fg):.3f}")


def split_dataset():
    names = sorted(p.stem for p in (DATA_DIR / "images").glob("*.png"))
    random.Random(CFG.SPLIT_SEED).shuffle(names)
    n = len(names)
    n_tr, n_va = round(CFG.SPLIT_RATIOS[0] * n), round(CFG.SPLIT_RATIOS[1] * n)
    split = {"train": names[:n_tr], "val": names[n_tr:n_tr + n_va], "test": names[n_tr + n_va:]}
    for pct in (2, 5, 10, 20):  # 2% dành cho thí nghiệm cực ít nhãn
        split[f"labeled_{pct}"] = split["train"][:int(len(split["train"]) * pct / 100)]
    SPLIT_DIR.mkdir(parents=True, exist_ok=True)
    for k, v in split.items():
        (SPLIT_DIR / f"{k}.txt").write_text("\n".join(v) + "\n", encoding="utf-8")
    print({k: len(v) for k, v in split.items()})
    return split


prepare_dataset()
SPLIT = split_dataset()

## 3. Tăng cường dữ liệu (Data Augmentation)

Chỉ áp dụng cho ảnh **huấn luyện có nhãn**; validation/test chỉ resize + chuẩn hóa.
Augmentation được áp dụng **trực tuyến** (online): mỗi lần đọc ảnh sinh ra một biến thể ngẫu nhiên
mới, nên với 30k vòng lặp, mô hình gần như không thấy hai ảnh giống hệt nhau – không cần sinh
thêm ảnh ra đĩa (offline).

| Nhóm | Phép biến đổi | Mô phỏng điều gì trong thực tế |
|---|---|---|
| Hình học | Lật ngang/dọc, xoay 90° | Tế bào không có hướng cố định |
| | Affine (co giãn, xoay nhỏ, tịnh tiến) | Khác độ phóng đại, vị trí đặt tiêu bản |
| | Elastic | Biến dạng hình thái tế bào nhẹ |
| Màu / nhuộm | Color jitter | Khác đèn, camera, cân bằng trắng |
| | **HED stain jitter** | Khác độ đậm nhạt thuốc nhuộm giữa các lam/phòng xét nghiệm |
| | Gamma, CLAHE | Khác độ tương phản, phơi sáng |
| Chất lượng | Gaussian blur, nhiễu Gauss | Lấy nét chưa chuẩn, nhiễu cảm biến |

**Mức `LEVEL`** – dùng để làm ablation về ảnh hưởng của augmentation:
- `"none"`: không tăng cường · `"basic"`: chỉ hình học · `"full"`: hình học + màu/nhuộm + chất lượng ảnh

In [ ]:
# ================================================================
#              CẤU HÌNH TĂNG CƯỜNG DỮ LIỆU (DATA AUGMENTATION)
#  p = xác suất áp dụng mỗi phép biến đổi cho một ảnh (0 = tắt, 1 = luôn áp dụng)
# ================================================================
class AUG:
    LEVEL = "full"            # "none" | "basic" | "full"

    # ---- Hình học (áp dụng đồng thời cho ảnh VÀ mask) ----
    HFLIP_P = 0.5             # lật ngang
    VFLIP_P = 0.5             # lật dọc
    ROT90_P = 0.5             # xoay 90/180/270 độ
    AFFINE_P = 0.5            # biến đổi affine:
    SCALE = (0.85, 1.15)      #   co giãn 0.85-1.15 lần (khác độ phóng đại)
    ROTATE = (-15, 15)        #   xoay thêm ±15 độ
    TRANSLATE = (-0.05, 0.05) #   tịnh tiến ±5% kích thước ảnh
    ELASTIC_P = 0.2           # biến dạng đàn hồi
    ELASTIC_ALPHA = 30        #   cường độ biến dạng (lớn -> méo nhiều; quá lớn làm tế bào sai hình)
    ELASTIC_SIGMA = 6         #   độ mượt của trường biến dạng

    # ---- Màu / nhuộm (CHỈ áp dụng cho ảnh, mask giữ nguyên) ----
    COLOR_JITTER_P = 0.5
    BRIGHTNESS = 0.2          # độ sáng ±20%
    CONTRAST = 0.2            # độ tương phản ±20%
    SATURATION = 0.2          # độ bão hòa ±20%
    HUE = 0.05                # sắc độ ±0.05 (giữ nhỏ để màu tế bào vẫn hợp lý)
    HED_P = 0.5               # dao động nhuộm trong không gian màu HED (Hematoxylin-Eosin-DAB);
    HED_SIGMA = 0.05          #   hệ số nhân mỗi kênh ~ U(1-σ, 1+σ)
    HED_BIAS = 0.005          #   độ lệch cộng mỗi kênh ~ U(-b, b). Giá trị HED của skimage chỉ
                              #   nằm trong ~[0, 0.1] nên b phải NHỎ; b = 0.02 đã làm nền trắng xóa.
                              #   (lam máu nhuộm Giemsa/Wright, HED là xấp xỉ hợp lý cho thuốc nhuộm
                              #    tím/hồng; đây là kỹ thuật chuẩn trong ảnh mô bệnh học)
    GAMMA_P = 0.2
    GAMMA = (80, 120)         # gamma 0.8-1.2 (đơn vị %)
    CLAHE_P = 0.1             # cân bằng histogram thích nghi (tăng tương phản cục bộ)

    # ---- Chất lượng ảnh ----
    BLUR_P = 0.1              # làm mờ Gauss (mô phỏng lệch tiêu cự)
    NOISE_P = 0.1             # nhiễu Gauss (mô phỏng nhiễu cảm biến)
    NOISE_STD = (0.01, 0.04)  #   độ lệch chuẩn nhiễu, tỉ lệ theo thang [0, 1] (0.04 ≈ ±10 mức xám)

    # ---- Chuẩn hóa ----
    # Thống kê ImageNet: bắt buộc cho Swin-Unet tiền huấn luyện; dùng chung cho UNeXt
    # để hai mô hình nhận đầu vào giống hệt nhau.
    MEAN = (0.485, 0.456, 0.406)
    STD = (0.229, 0.224, 0.225)

In [ ]:
# albumentations 1.x và 2.x đặt tên tham số khác nhau -> hai hàm trợ giúp để chạy được cả hai.
ALB_V2 = int(A.__version__.split(".")[0]) >= 2


def _affine():
    # Vùng trống sinh ra khi xoay/co được lấp bằng phản chiếu (không để viền đen giả).
    border = {"border_mode": cv2.BORDER_REFLECT_101} if ALB_V2 else {"mode": cv2.BORDER_REFLECT_101}
    return A.Affine(scale=AUG.SCALE, rotate=AUG.ROTATE, translate_percent=AUG.TRANSLATE,
                    p=AUG.AFFINE_P, **border)


def _noise():
    if ALB_V2:
        return A.GaussNoise(std_range=AUG.NOISE_STD, p=AUG.NOISE_P)
    var = tuple((s * 255) ** 2 for s in AUG.NOISE_STD)  # 1.x dùng phương sai theo thang 0-255
    return A.GaussNoise(var_limit=var, p=AUG.NOISE_P)


def hed_jitter(img, **kwargs):
    """Dao động nhuộm HED (Tellez et al., 2018): đổi RGB -> HED, nhân/cộng nhiễu từng kênh, đổi ngược."""
    hed = rgb2hed(img.astype(np.float32) / 255.0)
    alpha = np.random.uniform(1 - AUG.HED_SIGMA, 1 + AUG.HED_SIGMA, 3)
    beta = np.random.uniform(-AUG.HED_BIAS, AUG.HED_BIAS, 3)
    rgb = hed2rgb(hed * alpha + beta)
    return (np.clip(rgb, 0, 1) * 255).astype(np.uint8)


def build_train_transform(level=None, normalize=True):
    level = level or AUG.LEVEL
    t = [A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE)]
    if level in ("basic", "full"):
        t += [
            A.HorizontalFlip(p=AUG.HFLIP_P),
            A.VerticalFlip(p=AUG.VFLIP_P),
            A.RandomRotate90(p=AUG.ROT90_P),
            _affine(),
            A.ElasticTransform(alpha=AUG.ELASTIC_ALPHA, sigma=AUG.ELASTIC_SIGMA, p=AUG.ELASTIC_P),
        ]
    if level == "full":
        t += [
            A.ColorJitter(brightness=AUG.BRIGHTNESS, contrast=AUG.CONTRAST,
                          saturation=AUG.SATURATION, hue=AUG.HUE, p=AUG.COLOR_JITTER_P),
            A.Lambda(image=hed_jitter, p=AUG.HED_P),
            A.RandomGamma(gamma_limit=AUG.GAMMA, p=AUG.GAMMA_P),
            A.CLAHE(p=AUG.CLAHE_P),
            A.GaussianBlur(blur_limit=(3, 5), p=AUG.BLUR_P),
            _noise(),
        ]
    if normalize:
        t += [A.Normalize(mean=AUG.MEAN, std=AUG.STD), ToTensorV2()]
    return A.Compose(t)


def build_eval_transform():
    return A.Compose([A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
                      A.Normalize(mean=AUG.MEAN, std=AUG.STD), ToTensorV2()])


def show_augmentations(name=None, n=7, level=None):
    """Hiển thị ảnh gốc + n biến thể tăng cường; đường viền xanh lá = mask sau biến đổi."""
    name = name or SPLIT["train"][0]
    img = cv2.cvtColor(cv2.imread(str(DATA_DIR / "images" / f"{name}.png")), cv2.COLOR_BGR2RGB)
    msk = (cv2.imread(str(DATA_DIR / "masks" / f"{name}.png"), 0) > 127).astype(np.uint8)
    tf = build_train_transform(level, normalize=False)
    base = A.Compose([A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE)])(image=img, mask=msk)
    samples = [("Gốc", base["image"], base["mask"])]
    for i in range(n):
        o = tf(image=img, mask=msk)
        samples.append((f"Biến thể {i + 1}", o["image"], o["mask"]))
    fig, axes = plt.subplots(2, (n + 2) // 2, figsize=(3 * ((n + 2) // 2), 6))
    for ax, (title, im, m) in zip(axes.ravel(), samples):
        vis = im.copy()
        cnts, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        cv2.drawContours(vis, cnts, -1, (0, 255, 0), 1)
        ax.imshow(vis); ax.set_title(title); ax.axis("off")
    plt.suptitle(f"Data augmentation – LEVEL = {level or AUG.LEVEL}")
    plt.tight_layout(); plt.show()


show_augmentations()

Xem thử từng mức để so sánh (chạy lại ô để thấy biến thể ngẫu nhiên khác):

In [ ]:
show_augmentations(level="basic", n=3)

## 4. Dataset / DataLoader

In [ ]:
class BloodCellDataset(Dataset):
    def __init__(self, names, transform):
        self.names, self.transform = list(names), transform

    def __len__(self):
        return len(self.names)

    def __getitem__(self, i):
        name = self.names[i]
        img = cv2.cvtColor(cv2.imread(str(DATA_DIR / "images" / f"{name}.png")), cv2.COLOR_BGR2RGB)
        msk = (cv2.imread(str(DATA_DIR / "masks" / f"{name}.png"), 0) > 127).astype(np.uint8)
        o = self.transform(image=img, mask=msk)
        return {"image": o["image"], "mask": torch.as_tensor(o["mask"]).long(), "name": name}


def infinite_loader(loader):
    """Lặp DataLoader vô hạn (huấn luyện theo số vòng lặp)."""
    while True:
        for b in loader:
            yield b

## 5. Mô hình UNeXt

CNN kết hợp Tokenized/Shifted MLP (Valanarasu & Patel, MICCAI 2022).

In [ ]:
"""UNeXt: MLP-based Rapid Medical Image Segmentation Network (Valanarasu & Patel, MICCAI 2022).

Cài đặt lại theo repo gốc (jeya-maria-jose/UNeXt-pytorch), có thể cấu hình số kênh.
DACL dùng số kênh bộ mã hóa {32, 64, 128, 160, 256} (~1.47M tham số).

Kiến trúc:
  - 3 tầng tích chập (Conv -> BN -> MaxPool -> ReLU), mỗi tầng giảm kích thước 2 lần
  - 2 tầng "Tokenized MLP" (Shifted MLP) ở tầng sâu + nút cổ chai
  - Bộ giải mã đối xứng, nối tắt (skip connection) bằng phép CỘNG thay vì nối kênh
"""


class DWConv(nn.Module):
    """Tích chập theo chiều sâu 3x3 trên chuỗi token (mã hóa vị trí ngầm)."""

    def __init__(self, dim):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, 3, 1, 1, groups=dim)

    def forward(self, x, H, W):
        B, N, C = x.shape
        x = x.transpose(1, 2).reshape(B, C, H, W)
        x = self.dwconv(x)
        return x.flatten(2).transpose(1, 2)


class ShiftMLP(nn.Module):
    """Shifted MLP: dịch (shift) từng nhóm kênh theo chiều cao rồi chiều rộng trước mỗi lớp FC,
    giúp MLP "nhìn" được lân cận cục bộ giống tích chập nhưng rẻ hơn."""

    def __init__(self, in_features, hidden_features=None, out_features=None, drop=0.0, shift_size=5):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.dwconv = DWConv(hidden_features)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)
        self.shift_size = shift_size
        self.pad = shift_size // 2

    def _shift(self, x, H, W, dim):
        """Chia kênh thành shift_size nhóm, nhóm thứ k được cuộn (roll) k-pad pixel theo trục dim."""
        B, N, C = x.shape
        xn = x.transpose(1, 2).reshape(B, C, H, W)
        xn = F.pad(xn, (self.pad, self.pad, self.pad, self.pad), "constant", 0)
        chunks = torch.chunk(xn, self.shift_size, 1)
        shifted = [torch.roll(c, s, dim) for c, s in zip(chunks, range(-self.pad, self.pad + 1))]
        xs = torch.cat(shifted, 1)
        xs = torch.narrow(xs, 2, self.pad, H)
        xs = torch.narrow(xs, 3, self.pad, W)
        return xs.reshape(B, C, H * W).transpose(1, 2)

    def forward(self, x, H, W):
        x = self._shift(x, H, W, dim=2)  # dịch theo chiều cao
        x = self.fc1(x)
        x = self.dwconv(x, H, W)
        x = self.act(x)
        x = self.drop(x)
        x = self._shift(x, H, W, dim=3)  # dịch theo chiều rộng
        x = self.fc2(x)
        return self.drop(x)


class ShiftedBlock(nn.Module):
    def __init__(self, dim, mlp_ratio=1.0, drop=0.0, drop_path=0.0):
        super().__init__()
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = ShiftMLP(dim, int(dim * mlp_ratio), drop=drop)
        self.drop_path = DropPath(drop_path) if drop_path > 0 else nn.Identity()

    def forward(self, x, H, W):
        return x + self.drop_path(self.mlp(self.norm2(x), H, W))


class OverlapPatchEmbed(nn.Module):
    """Conv 3x3 stride 2 -> giảm kích thước 2 lần và chuyển feature map thành chuỗi token."""

    def __init__(self, in_chans, embed_dim, patch_size=3, stride=2):
        super().__init__()
        self.proj = nn.Conv2d(in_chans, embed_dim, patch_size, stride, patch_size // 2)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.proj(x)
        _, _, H, W = x.shape
        x = self.norm(x.flatten(2).transpose(1, 2))
        return x, H, W


class UNext(nn.Module):
    def __init__(self, num_classes=2, in_chans=3, dims=(32, 64, 128, 160, 256), drop_path_rate=0.0):
        """
        num_classes: số lớp đầu ra (2 = nền + tế bào).
        dims: số kênh 5 tầng mã hóa. 3 giá trị đầu cho tầng tích chập,
              2 giá trị cuối cho tầng Tokenized MLP và nút cổ chai.
              Bài báo UNeXt gốc: (16, 32, 128, 160, 256); DACL: (32, 64, 128, 160, 256).
        """
        super().__init__()
        c1, c2, c3, c4, c5 = dims
        # ---- Bộ mã hóa: tầng tích chập ----
        self.encoder1 = nn.Conv2d(in_chans, c1, 3, 1, 1)
        self.encoder2 = nn.Conv2d(c1, c2, 3, 1, 1)
        self.encoder3 = nn.Conv2d(c2, c3, 3, 1, 1)
        self.ebn1, self.ebn2, self.ebn3 = nn.BatchNorm2d(c1), nn.BatchNorm2d(c2), nn.BatchNorm2d(c3)
        # ---- Bộ mã hóa: Tokenized MLP ----
        self.patch_embed3 = OverlapPatchEmbed(c3, c4)
        self.patch_embed4 = OverlapPatchEmbed(c4, c5)
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, 3)]
        self.block1 = nn.ModuleList([ShiftedBlock(c4, drop_path=dpr[0])])
        self.block2 = nn.ModuleList([ShiftedBlock(c5, drop_path=dpr[1])])
        self.norm3, self.norm4 = nn.LayerNorm(c4), nn.LayerNorm(c5)
        # ---- Bộ giải mã ----
        self.dblock1 = nn.ModuleList([ShiftedBlock(c4, drop_path=dpr[0])])
        self.dblock2 = nn.ModuleList([ShiftedBlock(c3, drop_path=dpr[1])])
        self.dnorm3, self.dnorm4 = nn.LayerNorm(c4), nn.LayerNorm(c3)
        self.decoder1 = nn.Conv2d(c5, c4, 3, 1, 1)
        self.decoder2 = nn.Conv2d(c4, c3, 3, 1, 1)
        self.decoder3 = nn.Conv2d(c3, c2, 3, 1, 1)
        self.decoder4 = nn.Conv2d(c2, c1, 3, 1, 1)
        self.decoder5 = nn.Conv2d(c1, c1, 3, 1, 1)
        self.dbn1, self.dbn2 = nn.BatchNorm2d(c4), nn.BatchNorm2d(c3)
        self.dbn3, self.dbn4 = nn.BatchNorm2d(c2), nn.BatchNorm2d(c1)
        self.final = nn.Conv2d(c1, num_classes, 1)
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)

    @staticmethod
    def _to_map(x, B, H, W):
        return x.reshape(B, H, W, -1).permute(0, 3, 1, 2).contiguous()

    @staticmethod
    def _up(x):
        return F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)

    def forward(self, x):
        B = x.shape[0]
        # Bộ mã hóa - tầng tích chập (1/2, 1/4, 1/8)
        out = F.relu(F.max_pool2d(self.ebn1(self.encoder1(x)), 2, 2)); t1 = out
        out = F.relu(F.max_pool2d(self.ebn2(self.encoder2(out)), 2, 2)); t2 = out
        out = F.relu(F.max_pool2d(self.ebn3(self.encoder3(out)), 2, 2)); t3 = out
        # Tokenized MLP (1/16)
        out, H, W = self.patch_embed3(out)
        for blk in self.block1:
            out = blk(out, H, W)
        out = self._to_map(self.norm3(out), B, H, W); t4 = out
        # Nút cổ chai (1/32)
        out, H, W = self.patch_embed4(out)
        for blk in self.block2:
            out = blk(out, H, W)
        out = self._to_map(self.norm4(out), B, H, W)

        # Bộ giải mã
        out = F.relu(self._up(self.dbn1(self.decoder1(out)))) + t4
        _, _, H, W = out.shape
        out = out.flatten(2).transpose(1, 2)
        for blk in self.dblock1:
            out = blk(out, H, W)
        out = self._to_map(self.dnorm3(out), B, H, W)

        out = F.relu(self._up(self.dbn2(self.decoder2(out)))) + t3
        _, _, H, W = out.shape
        out = out.flatten(2).transpose(1, 2)
        for blk in self.dblock2:
            out = blk(out, H, W)
        out = self._to_map(self.dnorm4(out), B, H, W)

        out = F.relu(self._up(self.dbn3(self.decoder3(out)))) + t2
        out = F.relu(self._up(self.dbn4(self.decoder4(out)))) + t1
        out = F.relu(self._up(self.decoder5(out)))
        return self.final(out)  # logits (B, num_classes, H, W)

In [ ]:
def build_model(pretrained=True):
    """pretrained bị bỏ qua: UNeXt luôn huấn luyện từ đầu (giống bài báo)."""
    dims = {"dacl": (32, 64, 128, 160, 256), "original": (16, 32, 128, 160, 256)}[CFG.UNEXT_VARIANT]
    return UNext(num_classes=CFG.NUM_CLASSES, dims=dims)


_m = build_model()
print(f"UNeXt ({CFG.UNEXT_VARIANT}): {sum(p.numel() for p in _m.parameters()) / 1e6:.2f}M tham số")
print("Đầu ra:", tuple(_m(torch.randn(1, 3, CFG.IMG_SIZE, CFG.IMG_SIZE)).shape))
del _m

## 6. Hàm mất mát & chỉ số đánh giá

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.n = n_classes

    def forward(self, logits, target):
        prob = torch.softmax(logits, 1)
        oh = F.one_hot(target, self.n).permute(0, 3, 1, 2).float()
        inter = (prob * oh).sum((0, 2, 3))
        union = prob.sum((0, 2, 3)) + oh.sum((0, 2, 3))
        return 1 - ((2 * inter + 1e-5) / (union + 1e-5)).mean()


DICE = DiceLoss(CFG.NUM_CLASSES)


def sup_loss(logits, target):
    """L_sup = Cross-Entropy + Dice (giống DACL)."""
    return F.cross_entropy(logits, target) + DICE(logits, target)


def _surface(m):
    return m ^ ndimage.binary_erosion(m, structure=ndimage.generate_binary_structure(2, 1))


def binary_metrics(pred, gt):
    """DSC, Jaccard, HD95, ASD cho mask nhị phân. HD95/ASD tính theo PIXEL ở độ phân giải IMG_SIZE."""
    pred, gt = pred.astype(bool), gt.astype(bool)
    ps, gs = pred.sum(), gt.sum()
    if ps == 0 and gs == 0:
        return {"dsc": 1.0, "jaccard": 1.0, "hd95": 0.0, "asd": 0.0}
    inter = (pred & gt).sum()
    dsc, jac = 2 * inter / (ps + gs), inter / (pred | gt).sum()
    if ps == 0 or gs == 0:  # một bên rỗng -> phạt khoảng cách tối đa (đường chéo ảnh)
        d = float(np.hypot(*pred.shape))
        return {"dsc": float(dsc), "jaccard": float(jac), "hd95": d, "asd": d}
    sp, sg = _surface(pred), _surface(gt)
    d = np.concatenate([ndimage.distance_transform_edt(~sg)[sp], ndimage.distance_transform_edt(~sp)[sg]])
    return {"dsc": float(dsc), "jaccard": float(jac), "hd95": float(np.percentile(d, 95)), "asd": float(d.mean())}

## 7. Vòng huấn luyện

- Mỗi thí nghiệm lưu vào `runs/<model>_<ratio>p_s<seed>/`: `best.pth`, `last.pth`, `log.csv`, `config.json`, `test_per_image.csv`, `test_summary.json`.
- **Tự tiếp tục** từ `last.pth` nếu thí nghiệm đang dở; **tự bỏ qua** thí nghiệm đã có `test_summary.json`.
- Kết thúc mỗi thí nghiệm: nạp `best.pth` (Dice validation cao nhất) và đánh giá trên tập test.

In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


@torch.no_grad()
def evaluate(model, loader, full=False):
    model.eval()
    rows = []
    for b in loader:
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=CFG.USE_AMP and DEVICE.type == "cuda"):
            pred = model(b["image"].to(DEVICE)).argmax(1).cpu().numpy()
        for p, g, n in zip(pred, b["mask"].numpy(), b["name"]):
            if full:
                r = binary_metrics(p == 1, g == 1)
            else:
                s = (p == 1).sum() + (g == 1).sum()
                r = {"dsc": 1.0 if s == 0 else 2 * ((p == 1) & (g == 1)).sum() / s}
            r["name"] = n
            rows.append(r)
    model.train()
    return float(np.mean([r["dsc"] for r in rows])), rows


def run_test(model, out_dir):
    loader = DataLoader(BloodCellDataset(SPLIT["test"], build_eval_transform()), batch_size=8,
                        num_workers=CFG.NUM_WORKERS)
    _, rows = evaluate(model, loader, full=True)
    keys = ["dsc", "jaccard", "hd95", "asd"]
    pd.DataFrame(rows)[["name"] + keys].to_csv(out_dir / "test_per_image.csv", index=False)
    summary = {}
    for k in keys:
        v = np.array([r[k] for r in rows]) * (100 if k in ("dsc", "jaccard") else 1)  # DSC, Jaccard theo %
        summary[k] = {"mean": float(v.mean()), "std": float(v.std())}
    (out_dir / "test_summary.json").write_text(json.dumps(summary, indent=2))
    return summary


def train_one(ratio, seed):
    out_dir = RUN_ROOT / f"{CFG.MODEL}_{ratio}p_s{seed}"
    if (out_dir / "test_summary.json").exists():
        print(f"== {out_dir.name}: đã xong, bỏ qua ==")
        return json.loads((out_dir / "test_summary.json").read_text())
    out_dir.mkdir(parents=True, exist_ok=True)
    set_seed(seed)
    cfg_dump = {k: v for k, v in vars(CFG).items() if k.isupper()}
    cfg_dump.update({"AUG_" + k: v for k, v in vars(AUG).items() if k.isupper()})
    cfg_dump.update(ratio=ratio, seed=seed)
    (out_dir / "config.json").write_text(json.dumps(cfg_dump, indent=2, default=str))

    names = SPLIT["train"] if ratio == 100 else SPLIT[f"labeled_{ratio}"]
    if not names:
        raise ValueError(f"Mức {ratio}% không có ảnh có nhãn nào - kiểm tra LABELED_RATIOS / dữ liệu.")
    print(f"\n== {out_dir.name}: {len(names)} ảnh có nhãn, {CFG.MAX_ITERS} vòng lặp ==")
    g = torch.Generator(); g.manual_seed(seed)
    train_loader = DataLoader(BloodCellDataset(names, build_train_transform()),
                              batch_size=min(CFG.BATCH_SIZE, len(names)), shuffle=True, drop_last=True,
                              num_workers=CFG.NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
                              generator=g, persistent_workers=CFG.NUM_WORKERS > 0)
    val_loader = DataLoader(BloodCellDataset(SPLIT["val"], build_eval_transform()), batch_size=8,
                            num_workers=CFG.NUM_WORKERS)

    model = build_model().to(DEVICE)
    opt = torch.optim.SGD(model.parameters(), lr=CFG.BASE_LR, momentum=CFG.MOMENTUM,
                          weight_decay=CFG.WEIGHT_DECAY)
    use_amp = CFG.USE_AMP and DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=use_amp)

    start, best = 0, -1.0
    if (out_dir / "last.pth").exists():
        ck = torch.load(out_dir / "last.pth", map_location=DEVICE)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
        start, best = ck["iter"], ck["best"]
        print(f"   tiếp tục từ vòng {start} (best val Dice {best:.4f})")
    log_f = open(out_dir / "log.csv", "a", newline="")
    log = csv.writer(log_f)
    if start == 0:
        log.writerow(["iter", "lr", "loss", "val_dice"])

    model.train()
    it_loader = infinite_loader(train_loader)
    running, t0 = 0.0, time.time()
    for it in range(start, CFG.MAX_ITERS):
        b = next(it_loader)
        img, msk = b["image"].to(DEVICE, non_blocking=True), b["mask"].to(DEVICE, non_blocking=True)
        lr = CFG.BASE_LR * (1 - it / CFG.MAX_ITERS) ** 0.9
        for pg in opt.param_groups:
            pg["lr"] = lr
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=use_amp):
            logits = model(img)
        loss = sup_loss(logits.float(), msk)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update()
        running += loss.item()

        step = it + 1
        if step % CFG.LOG_INTERVAL == 0:
            print(f"   [{step:6d}/{CFG.MAX_ITERS}] loss {running / CFG.LOG_INTERVAL:.4f} | lr {lr:.5f} "
                  f"| {CFG.LOG_INTERVAL / (time.time() - t0):.1f} it/s")
            log.writerow([step, lr, running / CFG.LOG_INTERVAL, ""])
            running, t0 = 0.0, time.time()
        if step % CFG.VAL_INTERVAL == 0 or step == CFG.MAX_ITERS:
            vd, _ = evaluate(model, val_loader)
            log.writerow([step, "", "", vd]); log_f.flush()
            if vd > best:
                best = vd
                torch.save({"model": model.state_dict(), "iter": step, "val_dice": vd}, out_dir / "best.pth")
            print(f"   -> val Dice {vd:.4f} | best {best:.4f}")
            torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                        "iter": step, "best": best}, out_dir / "last.pth")
    log_f.close()

    model.load_state_dict(torch.load(out_dir / "best.pth", map_location=DEVICE)["model"])
    s = run_test(model, out_dir)
    print("   TEST:", "  ".join(f"{k.upper()} {v['mean']:.2f}({v['std']:.2f})" for k, v in s.items()))
    return s

## 8. Chạy thí nghiệm ở các mức nhãn

Ước lượng trên GPU T4: UNeXt ~25-40 phút / 30k vòng, Swin-Unet ~1.5-3 giờ / 30k vòng (ước lượng, chưa đo). Nếu không đủ 12 giờ, chia `LABELED_RATIOS` ra nhiều phiên.

In [ ]:
results = []
for ratio in CFG.LABELED_RATIOS:
    for seed in CFG.SEEDS:
        s = train_one(ratio, seed)
        results.append({"model": CFG.MODEL, "labeled_%": ratio, "seed": seed,
                        **{f"{k}_mean": v["mean"] for k, v in s.items()},
                        **{f"{k}_std": v["std"] for k, v in s.items()}})

## 9. Kết quả

In [ ]:
# ---- Bảng kết quả (định dạng giống bài báo: mean(std)) ----
df = pd.DataFrame(results)
table = pd.DataFrame({
    "Model": df["model"], "% nhãn": df["labeled_%"], "Seed": df["seed"],
    **{k.upper(): [f"{m:.2f}({s:.2f})" for m, s in zip(df[f"{k}_mean"], df[f"{k}_std"])]
       for k in ["dsc", "jaccard", "hd95", "asd"]}})
table.to_csv(RUN_ROOT / f"summary_{CFG.MODEL}.csv", index=False)
display(table)

# ---- Đường cong huấn luyện ----
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ratio in CFG.LABELED_RATIOS:
    log = pd.read_csv(RUN_ROOT / f"{CFG.MODEL}_{ratio}p_s{CFG.SEEDS[0]}" / "log.csv")
    tr, va = log.dropna(subset=["loss"]), log.dropna(subset=["val_dice"])
    axes[0].plot(tr["iter"], tr["loss"], label=f"{ratio}%")
    axes[1].plot(va["iter"], va["val_dice"], label=f"{ratio}%")
axes[0].set(title="Loss huấn luyện", xlabel="Vòng lặp"); axes[1].set(title="Dice validation", xlabel="Vòng lặp")
for ax in axes:
    ax.legend(title="Mức nhãn"); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(RUN_ROOT / f"curves_{CFG.MODEL}.png", dpi=150); plt.show()

In [ ]:
# ---- Ảnh dự đoán trên tập test: xanh lá = đúng, đỏ = dư (FP), xanh dương = thiếu (FN) ----
def show_predictions(ratio=None, n=4):
    ratio = ratio or CFG.LABELED_RATIOS[0]
    model = build_model(pretrained=False).to(DEVICE)
    ck = RUN_ROOT / f"{CFG.MODEL}_{ratio}p_s{CFG.SEEDS[0]}" / "best.pth"
    model.load_state_dict(torch.load(ck, map_location=DEVICE)["model"]); model.eval()
    ds = BloodCellDataset(SPLIT["test"][:n], build_eval_transform())
    fig, axes = plt.subplots(n, 3, figsize=(10, 3.3 * n))
    for i in range(n):
        b = ds[i]
        with torch.no_grad():
            p = model(b["image"][None].to(DEVICE)).argmax(1)[0].cpu().numpy().astype(bool)
        g = b["mask"].numpy().astype(bool)
        img = b["image"].permute(1, 2, 0).numpy() * np.array(AUG.STD) + np.array(AUG.MEAN)
        img = np.clip(img, 0, 1)
        err = np.zeros((*g.shape, 3)); err[p & g] = (0, 1, 0); err[p & ~g] = (1, 0, 0); err[~p & g] = (0, 0, 1)
        d = binary_metrics(p, g)["dsc"] * 100
        for ax, im, t in zip(axes[i], [img, g, err], ["Ảnh", "Ground truth", f"Dự đoán (DSC {d:.1f}%)"]):
            ax.imshow(im, cmap="gray"); ax.set_title(t); ax.axis("off")
    plt.suptitle(f"{CFG.MODEL} – {ratio}% nhãn"); plt.tight_layout(); plt.show()


show_predictions()

In [ ]:
# ---- Đóng gói kết quả (bỏ last.pth cho nhẹ) -> tải ở tab Output ----
zip_path = Path(CFG.WORK_DIR) / f"results_{CFG.MODEL}"
tmp = Path(CFG.WORK_DIR) / "_pack"
shutil.rmtree(tmp, ignore_errors=True)
shutil.copytree(RUN_ROOT, tmp, ignore=shutil.ignore_patterns("last.pth"))
shutil.make_archive(str(zip_path), "zip", tmp)
shutil.rmtree(tmp)
print("Đã tạo", zip_path.with_suffix(".zip"))